# Tutorial for the Quantum Phase Estimation Simulator Module

### Sergio A. Ortega and Daniel K. Park

In this notebook we show a tutorial of how to use the `qpe` module of our package SQWLib [[1]](https://arxiv.org/abs/2606.14226) for simulating the QPE algorithm applied to a Szegedy quantum walk [[2]](https://ieeexplore.ieee.org/document/1366222).

It is recommended to read section IV from [[1]](https://arxiv.org/abs/2606.14226) for terminology.

## Table of Contents

- [Packages](#0)
- [1 - Quantum Walk Preliminaries](#1)
- [2 - Operational Implementation](#2)
    - [2.1 - Initial Tensor State](#2-1)
    - [2.2 - Hadamard Gates](#2-2)
    - [2.3 - Controlled Gates](#2-3)
    - [2.4 - Quantum Fourier Transform](#2-4)
    - [2.5 - Other Operators](#2-5)
    - [2.6 - Unitary Operator](#2-6)
- [3 - Direct Implementation](#3)
- [4 - Measurement](#4)
    - [4.1 - Phase Measurement](#4-1)
    - [4.2 - Szegedy Measurement](#4-2)
- [5 - References](#5)

<a name='0'></a>
## Packages
First we import the libraries that we need, including our simulator.

In [1]:
import sys
sys.path.insert(0, '../')

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import sqwlib as sqw

<a name='1'></a>
## 1 - Quantum Walk Preliminaries

First, we have to define the unitary opertor $U$ of the quantum walk, and an initial Szegedy state. For this, we can use either the dense package SQUWALS [[3]](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/qute.202400022) or the sparse framework of our library.

For a tutorial on the SQUWALS package, see: https://github.com/OrtegaSA/squwals-repo/blob/main/Tutorials/SQUWALS%20Tutorial.ipynb.

For a tutorial on the sparse framework, see: ....

In [3]:
framework = 'dense'
framework = 'sparse'

N = 4

if framework == 'dense':
    
    from sqwlib import squwals as sw

    transition_matrix = np.array([[0. , 0.2, 0.2, 0. ],
                                  [0.5, 0.2, 0. , 0.5],
                                  [0.5, 0. , 0. , 0. ],
                                  [0. , 0.6, 0.8, 0.5]])
    
    fw = sw # fw is an alias for SQUWALS.

else: # sparse.
    
    weights_list = [
    (0,1,0.5), # Probability 0.5 from node 0 to node 1.
    (0,2,0.5),
    (1,0,0.2),
    (1,1,0.2),
    (1,3,0.6),
    (2,0,0.2),
    (2,3,0.8),
    (3,1,0.5),
    (3,3,0.5),
    ]
    
    sd = sqw.SpaceData(weights_list)
    
    transition_matrix = sd.obtain_transition_matrix(weights_list)
    
    fw = sd # fw is an alias for the SpaceData object.

As an example, we take $U = V^\dagger S V R_0$.

In [4]:
V = fw.Update(transition_matrix)
Vd = V.inverse()
R0 = fw.Reflection0()
S = fw.Swap()

U = Vd * S * V * R0
U.info()

Custom sparse unitary:
 1 - Sparse Reflection 0
 2 - Sparse Update
 3 - Sparse Swap
 4 - Update dagger


We create a Szegedy walk state.

In [5]:
sz_state = fw.create_no_coin_state(N)
sz_state.shape

(10,)

<a name='2'></a>
## 2 - Operational Implementation

In this section we show how to instantiate a state and construct the three main operators of the QPE circuit, as well as other operators of interest.

First, we need to choose the number of qubits in the phase register, as well as the number of repetitions of the phase register.

In [6]:
ph_qubits = 3
reps = 3

All operators take $reps=1$ if no argument is passed.

<a name='2-1'></a>
### 2.1 - Initial Tensor State

We use the function `create_qpe_tensor` to create the initial state for the QPE algorithm given a Szegedy walk state.

In [7]:
tensor_state = sqw.qpe.create_qpe_tensor(sz_state,ph_qubits,reps)
tensor_state.shape

(10, 8, 8, 8)

The walk state is in the first column of the tensor, and the rest of them are $0$.

In [8]:
tensor_state[:,0,0,0]

array([0.5, 0. , 0.5, 0. , 0. , 0.5, 0. , 0.5, 0. , 0. ])

In [9]:
tensor_state[:,0,0,1]

array([0., 0., 0., 0., 0., 0., 0., 0., 0., 0.])

<a name='2-2'></a>
### 2.2 - Hadamard Gates

For the Hadamard gates, we provide the class `Hadamard`. It requires as argument the number of repetitions.

In [10]:
H = sqw.qpe.Hadamard(reps)

<a name='2-3'></a>
### 2.3 - Controlled Gates

For the controlled gates, we provide the class `Controlled`. It requires as arguments the walk operator $U$, the number of phase qubits, and number of repetitions of the phase register.

In [11]:
C = sqw.qpe.Controlled(U,ph_qubits,reps)

<a name='2-4'></a>
### 2.4 - Quantum Fourier Transform

For the QFT and its inverse, we provide the classes `QFT` and `iQFT`. They require as argument the number of repetitions.

In [12]:
F = sqw.qpe.QFT(reps)
iF = sqw.qpe.IQFT(reps)

Moreover, one can be obtained from the other with the `inverse` method.

In [13]:
iF = F.inverse()
iF.info()

iQFT


In [14]:
F = iF.inverse()
F.info()

QFT


<a name='2-5'></a>
### 2.5 - Other Operators

Apart from the usual operator involved in the QPE algorithm, we may be intersted in other operators depending on the algorithm to study. For example, we may want to apply a particular walk operator to the walk register. Let us suposse we want to apply an oracle marking nodes in the first register. We first create the usual Szegedy operator with of our walk framework (SQUWALS or the sparse framework).

In [15]:
Q1 = fw.Oracle(1,marked_nodes=[0,2])

With it, we instantiate an object of the class `SzegedyOperator`, which applies it to a QPE tensor state.

In [16]:
Q1_qpe = sqw.qpe.SzegedyOperator(Q1)
Q1_qpe.info()

Sparse Oracle: Register 1: nodes [0, 2]


Other operator of interest is the reflection around the state $\left|0\right>$ in the phase register. For this, we provide the class `ReflectionPhase0`. No arguments are needed.

In [17]:
R0_phase = sqw.qpe.ReflectionPhase0()

<a name='2-6'></a>
### 2.6 - Unitary Operator

Now that we have seen how to create the building blocks, we can join them into a QPE unitary operator. Let us create the usual QPE operator

$$QFT^\dagger*C*H.$$

Note that the order of action of the operators is read from right to left. Thus, the first operator is the one of the Hadamard gates $H$.

We can create an unitary object with the `QPEUnitary` class, and use the `append` method to introduce the building blocks.

In [18]:
QPE = sqw.qpe.QPEUnitary()

# First we add the Hadamard operator.
QPE.append(H)

# Secondly the Controlled oeprator.
QPE.append(C)

# And so on
QPE.append(iF)

print(QPE)
QPE.info()

iFCH
Custom QPE unitary:
 1 - Hadamard
 2 - Controlled
 3 - iQFT


Another form of initializing the unitary is passing a list with the operators. The first operator of the list is the first one acting.

In [19]:
QPE = sqw.qpe.QPEUnitary(operators=[H,C,iF])
print(QPE)
QPE.info()

iFCH
Custom QPE unitary:
 1 - Hadamard
 2 - Controlled
 3 - iQFT


Finally, other form of constructing the unitary operator is multiplying the different operators in an algebraic sense, using `*`.

In [20]:
QPE = iF * C * H
print(QPE)
QPE.info()

iFCH
Custom QPE unitary:
 1 - Hadamard
 2 - Controlled
 3 - iQFT


In case we need to invert a QPE unitary operator, we can use the `inverse` method.

In [21]:
iQPE = QPE.inverse()
iQPE.info()

Custom QPE unitary:
 1 - QFT
 2 - Controlled
 3 - Hadamard


Finally, we can use the method `operate` passing a QPE tensor state, returning the action of the QPE unitary on it.

In [22]:
state_after_qpe = QPE.operate(tensor_state)
state_after_qpe.shape

(10, 8, 8, 8)

<a name='3'></a>
## 3 - Direct Implementation

Now, we show the direct implementation of the usual QPE circuit applied to a Szegedy walk state. We use the function `direct_qpe`, which takes as arguments:

- operator: A Szegedy unitary operator.
- sz_state: A Szegedy state.
- ph_qubits: Number of qubits of a single phase register.
- reps: Number of phase registers.

In [23]:
state_after_qpe = sqw.qpe.direct_qpe(U,sz_state,ph_qubits,reps)
state_after_qpe.shape

(10, 8, 8, 8)

<a name='4'></a>
## 4 - Measurement

Once the simulation has finished, we want to obtain some results from the resulting state. We can either obtain information from the phase registers, or from the walk register.

<a name='4-1'></a>
### 4.1 - Phase Measurement

We provide the class `PhaseMeasurement`. It requires as argument the number of repetitions.

In [24]:
ph_measurement = sqw.qpe.PhaseMeasurement(reps)

We obtain the probability distribution of all the phase registers considered as a whole with the `operate`method.

In [25]:
phase_distribution = ph_measurement.operate(state_after_qpe)
phase_distribution

array([2.09685934e-01, 3.71638848e-02, 8.38829225e-03, 4.62650867e-03,
       3.90767587e-03, 4.62650867e-03, 8.38829225e-03, 3.71638848e-02,
       3.71638848e-02, 9.52393075e-03, 2.71764338e-03, 9.78089848e-04,
       7.37595460e-04, 7.99445544e-04, 1.32461999e-03, 4.32885233e-03,
       8.38829225e-03, 2.71764338e-03, 2.01839985e-03, 3.57777345e-04,
       2.27685055e-04, 2.31615959e-04, 4.00312781e-04, 1.32461999e-03,
       4.62650867e-03, 9.78089848e-04, 3.57777345e-04, 1.22086728e-04,
       9.53922573e-05, 1.10532329e-04, 2.31615959e-04, 7.99445544e-04,
       3.90767587e-03, 7.37595460e-04, 2.27685055e-04, 9.53922573e-05,
       7.88026937e-05, 9.53922573e-05, 2.27685055e-04, 7.37595460e-04,
       4.62650867e-03, 7.99445544e-04, 2.31615959e-04, 1.10532329e-04,
       9.53922573e-05, 1.22086728e-04, 3.57777345e-04, 9.78089848e-04,
       8.38829225e-03, 1.32461999e-03, 4.00312781e-04, 2.31615959e-04,
       2.27685055e-04, 3.57777345e-04, 2.01839985e-03, 2.71764338e-03,
      

If we want to differenciate the results for each individual phase register, we can obtain a dictionary with the probability for each bitstring divided by registers. For this, we pass the argument `binary=True`.

In [26]:
phase_dict = ph_measurement.operate(state_after_qpe,binary=True)
phase_dict

{'000 000 000': 0.20968593373088298,
 '000 000 001': 0.03716388484793194,
 '000 000 010': 0.00838829224735385,
 '000 000 011': 0.00462650866936105,
 '000 000 100': 0.0039076758695748325,
 '000 000 101': 0.00462650866936105,
 '000 000 110': 0.00838829224735385,
 '000 000 111': 0.03716388484793194,
 '000 001 000': 0.03716388484793195,
 '000 001 001': 0.009523930747260833,
 '000 001 010': 0.002717643383478064,
 '000 001 011': 0.000978089847569168,
 '000 001 100': 0.0007375954598346561,
 '000 001 101': 0.0007994455440986503,
 '000 001 110': 0.0013246199920107307,
 '000 001 111': 0.004328852328829763,
 '000 010 000': 0.00838829224735385,
 '000 010 001': 0.002717643383478064,
 '000 010 010': 0.002018399849327166,
 '000 010 011': 0.0003577773447917037,
 '000 010 100': 0.0002276850545695408,
 '000 010 101': 0.00023161595857854262,
 '000 010 110': 0.00040031278138686255,
 '000 010 111': 0.0013246199920107307,
 '000 011 000': 0.00462650866936105,
 '000 011 001': 0.000978089847569168,
 '000 011 0

<a name='4-2'></a>
### 4.2 - Szegedy Measurement

For the walk, we have two registers. Thus, we first need to create the usual walk `Measurement` object of our walk framework (SQUWALS or the sparse framework), passing the desired walk register.

In [27]:
measurement = sd.Measurement(register=1)

With it, we instantiate an object of the class `SzegedyMeasurement`.

In [28]:
sz_measurement = sqw.qpe.SzegedyMeasurement(measurement)

Finally, we use the `operate` method on the QPE tensor state.

In [29]:
probability_distribution = sz_measurement.operate(state_after_qpe)
probability_distribution

array([0.23068959, 0.26563355, 0.19078334, 0.31289352])

<a name='5'></a>
## 5 - References

- [1] **[S. A. Ortega and D. K. Park. Efficient Simulation of Szegedy Quantum Walk Formulations and Algorithms. arXiv:2606.14226.](https://arxiv.org/abs/2606.14226)**
- [2] **[M. Szegedy. Quantum speed-up of Markov chain based algorithms. 45th Annual IEEE Symposium on Foundations of Computer Science, 32–41, 2004.](https://ieeexplore.ieee.org/document/1366222)**
- [3] **[S. A. Ortega and M. A. Martin-Delgado. SQUWALS: A Szegedy QUantum WALks Simulator. Advanced Quantum Technologies, 7:2400022, 2024.](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/qute.202400022)**